In [1]:
from google.colab import drive
import os
import urllib.request
import zipfile
import requests
from tqdm import tqdm
import pandas as pd
from os.path import join
from pycocotools.coco import COCO
import json

drive.mount('/content/drive')

# Install required packages
# !pip install pycocotools

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Create directory in Google Drive
drive_path = '/content/drive/MyDrive/COCO_Dataset'
os.makedirs(drive_path, exist_ok=True)

# COCO dataset URLs
urls = {
    'train_images': 'http://images.cocodataset.org/zips/train2017.zip',
    'val_images': 'http://images.cocodataset.org/zips/val2017.zip',
    'train_annotations': 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
}

# Download function with progress bar
def download_file(url, destination):

    """Download file with progress bar"""

    class DownloadProgressBar(tqdm):

        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)

    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=url.split('/')[-1]) as t:
        urllib.request.urlretrieve(url, filename=destination, reporthook=t.update_to)

# Download and extract each file
for name, url in urls.items():
    print(f"\n{'='*50}")
    print(f"Downloading {name}...")
    print(f"{'='*50}")

    zip_path = os.path.join(drive_path, f'{name}.zip')

    # Download
    if not os.path.exists(zip_path):
        download_file(url, zip_path)
        print(f"✓ Downloaded {name}")
    else:
        print(f"✓ {name} already exists, skipping download")

    # Extract
    print(f"Extracting {name}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(drive_path)
    print(f"✓ Extracted {name}")

    # Optional: Remove zip file to save space
    # os.remove(zip_path)

print("\n" + "="*50)
print("✓ COCO dataset download complete!")
print(f"Location: {drive_path}")
print("="*50)


✓ train_images already exists, skipping download
Extracting train_images...
✓ Extracted train_images

✓ val_images already exists, skipping download
Extracting val_images...
✓ Extracted val_images

✓ train_annotations already exists, skipping download
Extracting train_annotations...
✓ Extracted train_annotations

✓ COCO dataset download complete!
Location: /content/drive/MyDrive/COCO_Dataset


In [3]:
# Display dataset structure
print("\nDataset structure:")
for root, dirs, files in os.walk(drive_path):
    level = root.replace(drive_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:3]:  # Show first 3 files
        print(f'{subindent}{file}')
    if len(files) > 3:
        print(f'{subindent}... and {len(files)-3} more files')



# Load instance annotations (contains mask annotations)
instances_path = os.path.join(drive_path, 'annotations/instances_train2017.json')
coco = COCO(instances_path)

print("\n" + "="*50)
print("Annotation Statistics:")
print("="*50)
print(f"Total images: {len(coco.imgs)}")
print(f"Total annotations: {len(coco.anns)}")
print(f"Total categories: {len(coco.cats)}")

# Show some categories
cats = coco.loadCats(coco.getCatIds())
print("\nSample categories:")
for cat in cats[:10]:
    print(f"  - {cat['name']}")


Dataset structure:
COCO_Dataset/
  train_images.zip
  val_images.zip
  train_annotations.zip
  train2017/
    000000147328.jpg
    000000414738.jpg
    000000281563.jpg
    ... and 118284 more files
  val2017/
    000000212226.jpg
    000000231527.jpg
    000000578922.jpg
    ... and 4997 more files
  annotations/
    instances_train2017.json
    instances_val2017.json
    captions_train2017.json
    ... and 3 more files
loading annotations into memory...
Done (t=18.83s)
creating index...
index created!

Annotation Statistics:
Total images: 118287
Total annotations: 860001
Total categories: 80

Sample categories:
  - person
  - bicycle
  - car
  - motorcycle
  - airplane
  - bus
  - train
  - truck
  - boat
  - traffic light


In [4]:
# Initialize COCO API
coco = COCO(os.path.join(drive_path, 'annotations/instances_val2017.json'))
print(f"COCO dataset loaded: {len(coco.getImgIds())} images")

loading annotations into memory...
Done (t=1.18s)
creating index...
index created!
COCO dataset loaded: 5000 images
